# VERGIL RL Training via GRPO (Google Colab)

This notebook trains a 0.5B parameter LLM (Qwen2.5) to act as the VERGIL agent using **Group Relative Policy Optimization (GRPO)** and **Unsloth** for 4-bit memory efficient training.

**Hardware Requirements:** T4 GPU (free tier Colab is sufficient).

## 1. Setup Environment

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install gymnasium networkx datasets

In [ ]:
# Clone the VERGIL repository to access the environment and CDG engine
!rm -rf Vergil
!git clone https://github.com/laksh718/Vergil.git
%cd Vergil

import sys
sys.path.insert(0, '.')

In [ ]:
import torch
import torch.utils._pytree
import warnings

# ── ULTIMATE STABILITY PATCHES ──
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")
for i in range(1, 8):
    if not hasattr(torch, f'int{i}'): setattr(torch, f'int{i}', torch.int8)
if not hasattr(torch.utils._pytree, 'register_constant'):
    torch.utils._pytree.register_constant = lambda cls: cls

## 2. Load VERGIL Engine and Reward Logic

In [ ]:
from scripts.train_grpo_colab import (
    VERGILEnv, POMDPWrapper, CurriculumEngine, ScenarioGenerator, FailureTopologyDatabase,
    state_to_prompt, vergil_reward_function
)

print("🌍 Initializing VERGIL environment...")
env = VERGILEnv(seed=42)
pomdp = POMDPWrapper(env)
failure_db = FailureTopologyDatabase(db_path='/tmp/vergil_ftd_grpo.sqlite')
curriculum = CurriculumEngine(failure_db=failure_db, scenario_generator=ScenarioGenerator(seed=42))

def reward_fn(prompts, completions, **kw):
    return vergil_reward_function(prompts, completions, env=env, pomdp=pomdp, num_generations=4)

## 3. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=64, lora_alpha=128, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

if not hasattr(model, "warnings_issued"): model.warnings_issued = {}

## 4. Train with GRPO

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# Pre-generate small prompt set for T4 speed
training_prompts = []
for _ in range(50):
    vs, _, _ = pomdp.reset(scenario=curriculum.generate_next_episode())
    training_prompts.append(state_to_prompt(vs, env))

config = GRPOConfig(
    output_dir="/tmp/vergil_grpo_output",
    max_steps=50, 
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5, 
    max_completion_length=512, 
    num_generations=4, 
    logging_steps=5,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none"
)

trainer = GRPOTrainer(
    model=model, args=config, 
    train_dataset=Dataset.from_dict({"prompt": training_prompts}),
    reward_funcs=[reward_fn], processing_class=tokenizer
)

trainer.train()